# nmrmetaproc — Real Data Demo

**Folorunsho Bright Omage** | DOI: [10.5281/zenodo.19194107](https://doi.org/10.5281/zenodo.19194107)

Process real Bruker ¹H NMR FID data from a clinical metabolomics study.

This notebook demonstrates the complete workflow using de-identified NMR data from a thrombosis (VTE) metabolomics study. The data includes 3 Control and 3 Thrombosis samples acquired on a 600 MHz Bruker spectrometer using the noesygppr1d pulse sequence.

In [ ]:
# Install nmrmetaproc package
!pip install nmrmetaproc -q

In [ ]:
# Download example data from the GitHub repository
!git clone --depth 1 https://github.com/omagebright/nmrmetaproc.git /content/nmrmetaproc_repo
import shutil
shutil.copytree('/content/nmrmetaproc_repo/example_data', '/content/example_data')
print("Example data downloaded successfully!")

## Dataset Description

The example dataset contains de-identified NMR data from a clinical metabolomics study investigating thrombosis (VTE):

- **Samples**: 3 Control + 3 Thrombosis patients
- **Instrument**: 600 MHz Bruker AVANCE III
- **Pulse sequence**: noesygppr1d (1D NOESY with presaturation)
- **Solvent**: D₂O with TSP internal standard
- **Temperature**: 298 K

Each sample contains:
- `fid`: Raw FID data (131,072 complex points)
- `acqus`: Acquisition parameters
- `pdata/1/procs`: Processing parameters

In [ ]:
# Initialize NMRProcessor and run the full processing pipeline
from nmrmetaproc import NMRProcessor
import matplotlib.pyplot as plt
import numpy as np

# Configure processing parameters
proc = NMRProcessor(
    lb=0.3,                    # Line broadening: 0.3 Hz
    bin_width=0.01,            # Spectral binning: 0.01 ppm
    ppm_range=(0.5, 9.5),      # Chemical shift range
    normalization='pqn',        # Probabilistic Quotient Normalization
    snr_threshold=10,          # Signal-to-noise threshold
    linewidth_threshold=5.0    # Linewidth threshold for QC
)

# Process the example data
results = proc.process('/content/example_data')
results.save('/content/output')

print(f"Processing completed!")
print(f"Total samples: {results.n_total}")
print(f"Passed QC: {results.n_passed}")
print(f"Failed QC: {results.n_failed}")

if results.n_passed > 0:
    print(f"Spectral matrix shape: {results.spectral_matrix.shape}")
    print(f"Chemical shift range: {results.ppm_axis[0]:.2f} to {results.ppm_axis[-1]:.2f} ppm")

In [ ]:
# Display QC report table
import pandas as pd

qc_data = []
for i, (sample_id, passed) in enumerate(zip(results.sample_ids, results.qc_passed)):
    qc_data.append({
        'Sample ID': sample_id,
        'QC Status': 'PASS' if passed else 'FAIL',
        'Group': 'Control' if 'Control' in sample_id else 'Thrombosis'
    })

df = pd.DataFrame(qc_data)
print("Quality Control Summary:")
print("=" * 40)
display(df)

In [ ]:
# Plot all spectra overlaid (Control=blue, Thrombosis=red) - Publication quality
plt.style.use('default')
fig, ax = plt.subplots(figsize=(14, 8), dpi=150)

# Define colors for groups
colors = {'Control': '#2E86AB', 'Thrombosis': '#A23B72'}

# Plot spectra
for i, (sample_id, spectrum, passed) in enumerate(zip(results.sample_ids, results.spectra, results.qc_passed)):
    if passed:  # Only plot samples that passed QC
        group = 'Control' if 'Control' in sample_id else 'Thrombosis'
        color = colors[group]
        
        # Add small offset for visibility
        offset = i * 0.05 if group == 'Control' else i * 0.05 + 0.3
        
        ax.plot(results.ppm_axis, spectrum + offset, color=color, alpha=0.8, linewidth=1.2,
                label=group if sample_id.endswith('01') else "")

# Formatting
ax.set_xlim(9.5, 0.5)  # Inverted x-axis (standard for NMR)
ax.set_xlabel('Chemical Shift (ppm)', fontsize=14, fontweight='bold')
ax.set_ylabel('Intensity (a.u.)', fontsize=14, fontweight='bold')
ax.set_title('Processed ¹H NMR Spectra — Control vs Thrombosis (VTE)', fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=12, frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Displayed {sum(results.qc_passed)} spectra that passed quality control.")

## Citation & Links

If you use **nmrmetaproc** in your research, please cite:

```
Omage, F.B. (2026). nmrmetaproc: Automated NMR metabolomics data processing pipeline. 
Zenodo. https://doi.org/10.5281/zenodo.19194107
```

### Links
- **GitHub Repository**: https://github.com/omagebright/nmrmetaproc
- **PyPI Package**: https://pypi.org/project/nmrmetaproc/
- **Author ORCID**: [0000-0002-9750-5034](https://orcid.org/0000-0002-9750-5034)

### Key Features Demonstrated
✅ **Automated processing** of Bruker FID data<br>
✅ **Quality control** with SNR and linewidth filtering<br>
✅ **Phase correction** using ACME algorithm<br>
✅ **Baseline correction** with asymmetric least squares<br>
✅ **Spectral alignment** using icoshift<br>
✅ **Normalization** with probabilistic quotient normalization (PQN)<br>
✅ **Spectral binning** with customizable resolution<br>
✅ **Batch processing** for high-throughput studies

---

*This demo uses real, de-identified clinical data to showcase the package capabilities.*